# 🥇 Tahap Final SOTA: NVIDIA StyleGAN2-ADA
Notebook ini membuang arsitektur Keras kustom yang menemui jalan buntu dan beralih ke Teknologi *State-of-The-Art* resmi NVIDIA yang dirancang khusus untuk dataset mikroskopis (< 1.000 gambar). Arsitektur PyTorch ini dijamin memecahkan rekor FID skor di atas angin!

In [1]:
# 1. SETUP LINGKUNGAN KERJA NVIDIA
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch

!pip install ninja

# ====== PATCH TOTAL PYTORCH 2.x + CUDA 12.x ======
import os, re

# FIX 1: InfiniteSampler
with open('torch_utils/misc.py', 'r') as f:
    content = f.read()
content = content.replace('super().__init__(dataset)', 'super().__init__()')
with open('torch_utils/misc.py', 'w') as f:
    f.write(content)
print('[FIX 1] InfiniteSampler - OK')

# FIX 2: custom_ops.py
with open('torch_utils/custom_ops.py', 'r') as f:
    lines = f.readlines()
new_lines = []
i = 0
while i < len(lines):
    line = lines[i]
    if 'torch.utils.cpp_extension.load(' in line and 'module =' not in line:
        new_lines.append(line.replace('torch.utils.cpp_extension.load(', 'module = torch.utils.cpp_extension.load('))
        if i + 1 < len(lines) and 'importlib.import_module' in lines[i + 1]:
            i += 2
            continue
    elif 'importlib.import_module(module_name)' in line:
        i += 1
        continue
    else:
        new_lines.append(line)
    i += 1
with open('torch_utils/custom_ops.py', 'w') as f:
    f.writelines(new_lines)
with open('torch_utils/custom_ops.py', 'r') as f:
    c = f.read()
    if 'importlib.import_module(module_name)' in c:
        c = c.replace('        module = importlib.import_module(module_name)', '        pass')
        with open('torch_utils/custom_ops.py', 'w') as fw:
            fw.write(c)
print('[FIX 2] custom_ops.py - OK')

# FIX 3: conv2d_gradfix.py
CONV2D_GRADFIX = '''import contextlib
import torch
enabled = False
weight_gradients_disabled = False
@contextlib.contextmanager
def no_weight_gradients():
    global weight_gradients_disabled
    old = weight_gradients_disabled
    weight_gradients_disabled = True
    yield
    weight_gradients_disabled = old
def conv2d(input, weight, bias=None, stride=1, padding=0, dilation=1, groups=1):
    return torch.nn.functional.conv2d(input=input, weight=weight, bias=bias, stride=stride, padding=padding, dilation=dilation, groups=groups)
def conv_transpose2d(input, weight, bias=None, stride=1, padding=0, output_padding=0, groups=1, dilation=1):
    return torch.nn.functional.conv_transpose2d(input=input, weight=weight, bias=bias, stride=stride, padding=padding, output_padding=output_padding, groups=groups, dilation=dilation)
'''
with open('torch_utils/ops/conv2d_gradfix.py', 'w') as f:
    f.write(CONV2D_GRADFIX)
print('[FIX 3] conv2d_gradfix.py - OK')

# FIX 4: grid_sample_gradfix.py
GRID_SAMPLE = '''import torch
enabled = False
def grid_sample(input, grid):
    return torch.nn.functional.grid_sample(input=input, grid=grid, mode="bilinear", padding_mode="zeros", align_corners=False)
'''
with open('torch_utils/ops/grid_sample_gradfix.py', 'w') as f:
    f.write(GRID_SAMPLE)
print('[FIX 4] grid_sample_gradfix.py - OK')

# FIX 5: loss.py R1 - bypass augment_pipe
with open('training/loss.py', 'r') as f:
    lines = f.readlines()
new_lines = []
patched = False
for i, line in enumerate(lines):
    if 'r1_grads = torch.autograd.grad' in line and 'real_img_tmp' in line:
        insert_idx = None
        for j in range(len(new_lines) - 1, max(0, len(new_lines) - 8), -1):
            if 'if do_Dr1:' in new_lines[j]:
                insert_idx = j + 1
                break
        if insert_idx is not None:
            indent = '                    '
            new_lines.insert(insert_idx, indent + '# PATCH: R1 bypass augmentation\n')
            new_lines.insert(insert_idx + 1, indent + 'real_img_r1 = real_img_tmp.detach().requires_grad_(True)\n')
            new_lines.insert(insert_idx + 2, indent + 'with misc.ddp_sync(self.D, sync):\n')
            new_lines.insert(insert_idx + 3, indent + '    real_logits_r1 = self.D(real_img_r1, real_c)\n')
            line = line.replace('real_img_tmp', 'real_img_r1')
            line = line.replace('real_logits.sum()', 'real_logits_r1.sum()')
            new_lines.append(line)
            patched = True
        else:
            new_lines.append(line)
    else:
        new_lines.append(line)
if patched:
    with open('training/loss.py', 'w') as f:
        f.writelines(new_lines)
    print('[FIX 5] loss.py - R1 bypass augment_pipe - OK')
else:
    print('[FIX 5] ERROR: r1_grads pattern not found!')

# FIX 6: Warning suppression
import site
site_dir = site.getsitepackages()[0] if site.getsitepackages() else site.getusersitepackages()
with open(os.path.join(site_dir, 'usercustomize.py'), 'w') as f:
    f.write('import warnings\nwarnings.filterwarnings("ignore")\n')
print('[FIX 6] Warning suppression - OK')

# FIX 7: DDP consistency check - noise_const
with open('training/training_loop.py', 'r') as f:
    tl = f.read()
tl = tl.replace(
    "ignore_regex=r'.*\\.w_avg'",
    "ignore_regex=r'.*\\.(w_avg|noise_const|noise_strength)'"
)
with open('training/training_loop.py', 'w') as f:
    f.write(tl)
print('[FIX 7] training_loop.py - DDP ignore noise_const - OK')

print('\n=== SEMUA 7 PATCH BERHASIL ===')


In [2]:
# 2. KOMPRESI DATASET UNTUK ALGORITMA NVIDIA
import os

# Pastikan path ini benar merujuk ke folder gambar .jpg / .png murni
INPUT_DIR = "/kaggle/input/datasets/mathallahrafifaldera/datserviks/dabar"
OUTPUT_ZIP = "/kaggle/working/serviks_128.zip"

!python dataset_tool.py --source $INPUT_DIR --dest $OUTPUT_ZIP --width 128 --height 128


  0%|                                                   | 0/666 [00:00<?, ?it/s]/kaggle/working/stylegan2-ada-pytorch/dataset_tool.py:429: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PIL.Image.fromarray(img, { 1: 'L', 3: 'RGB' }[channels])
100%|█████████████████████████████████████████| 666/666 [01:54<00:00,  5.84it/s]


In [3]:
# 3. TRAINING + VISUALISASI OTOMATIS (DENGAN AUTO-RESUME)
import os, warnings
warnings.filterwarnings('ignore')

OUT_DIR = "/kaggle/working/results"

# --- FITUR PENCARIAN DATASET PINTAR ---
zip_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.zip'): zip_files.append(os.path.join(root, file))
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if file.endswith('.zip'): zip_files.append(os.path.join(root, file))

target_zips = [f for f in zip_files if 'serviks' in f.lower()]
if not target_zips:
    print('Peringatan: File ZIP dataset (serviks_128.zip) tidak ditemukan!')
    DATASET_ZIP = ""
else:
    DATASET_ZIP = target_zips[0]
    print(f'\nDATASET ZIP DITEMUKAN: {DATASET_ZIP}')

os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['TORCH_CPP_LOG_LEVEL'] = 'ERROR'
os.environ['NCCL_DEBUG'] = 'ERROR'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# --- FITUR AUTO-RESUME SUPER PINTAR (OS.WALK) ---
all_pkl = []
print("\n" + "*"*60)
print("--- MENCARI CHECKPOINT PKL DI SELURUH FOLDER KAGGLE ---")

# Cari di /kaggle/input (tempat dataset upload berada)
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.pkl'):
            found_pkl = os.path.join(root, file)
            all_pkl.append(found_pkl)
            print(f"-> Ditemukan: {found_pkl}")

# Cari di /kaggle/working (tempat output training berjalan)
for root, dirs, files in os.walk('/kaggle/working'):
    for file in files:
        if file.endswith('.pkl'):
            found_pkl = os.path.join(root, file)
            all_pkl.append(found_pkl)
            print(f"-> Ditemukan: {found_pkl}")

resume_cmd = ""
if all_pkl:
    latest_pkl = sorted(all_pkl)[-1]
    print(f"\nCHECKPOINT BERHASIL DIPILIH!")
    print(f"Melanjutkan training dari: {latest_pkl}")
    resume_cmd = f'--resume="{latest_pkl}"'
else:
    print(f"\nCHECKPOINT TIDAK DITEMUKAN!")
    print(f"Sesi ini akan memulai training dari awal (nol).")
    print(f"PASTIKAN Anda sudah menekan tombol 'Add Data' di Kaggle dan memilih file PKL Anda.")
print("*"*60 + "\n")

# RUN TRAINING
cmd = f"python -W ignore train.py --outdir={OUT_DIR} --data={DATASET_ZIP} --gpus=2 --cfg=stylegan2 --metrics=fid50k_full --kimg=4000 --snap=25 --aug=ada --target=0.6 --batch=32 {resume_cmd}"
print(f'\n[EXEC] {cmd}\n')
os.system(cmd)
